In [40]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as mlt
import math as Math

In [41]:
class Value:
    def __init__(self,data,_children=(),op='',label=''):
        self.data=data
        self.back=set(_children)
        self._backward= lambda : None
        self.op=op
        self.grad=0
        self.label=label
    def __repr__(self):
        return f"value ({self.data})"
    
    def __add__(self,other):
        
        other = other if isinstance(other,Value) else Value(other)
        out =Value((self.data+other.data),(self,other))
       
        def _backward():
            self.grad+=1*out.grad
            other.grad+=1*out.grad
        out._backward=_backward
        return out
    
    def __mul__(self,other):
        other = other if isinstance(other,Value) else Value(other)
        out =Value((self.data*other.data),(self,other))
        
        def _backward():
            self.grad+=other.data*out.grad
            other.grad+=self.data*out.grad
        out._backward=_backward
        return out
  
    def printgrads(self):
        print(self.label,self.grad)
        for child in self.back:
            Value.printgrads(child)
    
    def backk(self):
        topo=[]
        self.grad=1
        visited=set()
        def build_topo(v):
            if v  not in visited:
                visited.add(v)
            for child in v.back:
                build_topo(child)
            topo.append(v)
        build_topo(self)
        for node in reversed(topo):
            node._backward()
    def __rmul__(self,other):
        return self*other

    def tanh(self):
        out=Value(((Math.exp(2*self.data))-1)/(Math.exp(2*self.data)+1),(self,),'tanh')
        def _backward():
            self.grad=1-(out.data**2)
        out._backward=_backward
        return out


In [42]:
def undBackProp():
    a=Value(2,label='a')
    b=Value(5,label='b')
    c=Value(-0.5,label='c')
    f=Value(0.7,label='f')
    e=a*b
    d=c+e
    l=d*f
    print(l)

    a=Value(2,label='a')
    b=Value(5,label='b')
    c=Value(-0.5,label='c')
    f=Value(0.7,label='f')
    e=a*b
    d=c+e
    l=d*f
    print(l)
undBackProp()
    

value (6.6499999999999995)
value (6.6499999999999995)


In [43]:
#Actual Neuron which have weights and biases will later do it on xor
x1=Value(2,label='x1')
x2=Value(0,label='x2')

w1=Value(-3,label='w1')
w2=Value(1,label='w2')

b=Value(6.8813735870195432,label='b')

x1w1=x1*w1
x2w2=x2*w2
n1=x1w1+x2w2
n=x1w1+x2w2+b
o=n.tanh()
print(o)
o.backk()
o.printgrads()
2*a

value (0.7071067811865476)
 1
 0.4999999999999999
b 0.4999999999999999
 0.4999999999999999
 0.4999999999999999
x1 -1.4999999999999996
w1 0.9999999999999998
 0.4999999999999999
x2 0.4999999999999999
w2 0.0


value (6)

In [ ]:
#Backpropping and giving grads manually without using made func
o.grad=1
print(o.grad)
n.grad=1-(o.data**2)
b.grad=n.grad
print(n.grad)

In [ ]:
x1w1.grad=n.grad
x2w2.grad=n.grad
x1.grad=w1.data*x1w1.grad
w1.grad=x1.data*x1w1.grad
x2.grad=w2.data*x2w2.grad
w2.grad=x2.data*x2w2.grad
o.printgrads()
